In [ ]:
from aligner.networksHandling import helper as helper_networks
from aligner.poissonAlignment import helper as helper_poisson
from aligner.montecarlo import helper as helper_montecarlo
from aligner.analysis import helper as helper_analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import pickle
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import r2_score
import matplotlib.ticker as ticker
import math

In [ ]:
#create directories to store results (if they don't exist)
!mkdir -p results/

In [ ]:
#functions.........

def remove_At_onwards(A, t_study):
    #just keep the previous networks:
    T, N, N = A.shape
    t_in = [t for t in range(T) if t<t_study]

    subsetA = A.copy()
    subsetA = subsetA[t_in]
    
    return subsetA

def alignment_withprevious(A, t_study, groups, R):
    T, N, N = A.shape
    t_in = [t for t in range(T) if t<t_study]

    subsetA = A.copy()
    subsetA = subsetA[t_in]
    T = subsetA.shape[0]

    #initial permutations:
    P = helper_montecarlo.initial_permutation_groups(subsetA, groups) #nodes are ordered by strength

    #gt permutation
    gt_P = [np.arange(N) for t in range(T)]

    #alignment:
    betas = np.array([beta_spacing**n for n in range(-12,1)])
    B = len(betas)
    Egt = helper_poisson.calculateEnergy_total(subsetA, gt_P, helper_poisson.get_Salpha(subsetA), N, T)
    
    all_results = []
    for run in range(R):
        start = time.process_time()

        Esweeps_r, samples, groundState, MCacceptance_rate = helper_montecarlo.sampling_poisson_groups(
            subsetA, P, betas, 10000, 5000, 2, groups, "")

        end = time.process_time()
        print(f"Process time (1 run, s): {end-start}")
        results = dict({"E": Esweeps_r, "E_GT": Egt, "GS": groundState, "Samples": samples})
        with open(f"connectomes_upto_{t_study}_run{run}.pkl", "wb") as file:
            pickle.dump(results, file)
    
        all_results.append(results)
            
        for b in range(B):
            plt.plot(Esweeps_r[b], label=round(betas[b], 3))
        plt.plot([Egt for _ in range(len(Esweeps_r[0]))], color="red", linestyle="dashed", label = "GT Energy")
        plt.legend(title=r"$\beta$", bbox_to_anchor=(1, 1))
        plt.show()

    return all_results

def invperm(perm):

    N = len(perm)
    inv_perm = np.empty(N, dtype=int)
    for i in range(N):
        inv_perm[perm[i]] = i
    return inv_perm

def latent_assignment_seqthreshold(ensemble):

    T = len(ensemble[0])
    N = len(ensemble[0][0])
    latent_assignments = []  # stores one latent vector per sampled alignment
    
    for alignment in ensemble:
        latent = [None] * N
        used_labels = set()
    
        # Sequential assignment by decreasing threshold
        for threshold in range(T, 0, -1):
            for n in range(N):
                if latent[n] is not None:
                    continue  # already assigned
                
                this_column = [alignment[t][n] for t in range(T)]
                values, counts = np.unique(this_column, return_counts=True)
    
                # candidate labels with enough frequency and not used
                candidates = [val for val, count in zip(values, counts)
                              if count >= threshold and val not in used_labels]
    
                if candidates:
                    # choose the most frequent available label
                    chosen_value = candidates[np.argmax([counts[np.where(values == val)[0][0]] for val in candidates])]
                    latent[n] = chosen_value
                    used_labels.add(chosen_value)
    
        # Fill unassigned nodes with remaining unused labels
        missing_labels = [val for val in range(N) if val not in used_labels]
        for i in range(N):
            if latent[i] is None:
                latent[i] = missing_labels.pop(0)
                used_labels.add(latent[i])
    
        # Save final latent assignment for this sample
        latent_assignments.append(latent)
    
    latent_assignments = np.array(latent_assignments)
    return latent_assignments
    
def infer_blueprint(A, ensemble):

    T = len(ensemble[0])
    N = len(ensemble[0][0])

    # Compute latent assignments from ensemble
    ensemble_latent = latent_assignment_seqthreshold(ensemble)

    # Store reordered L-matrices
    L_values = []

    for idx, alignment in enumerate(ensemble):
        alignment = np.rint(alignment).astype(int) #ensure int np type
        latent = ensemble_latent[idx]
        L = helper_poisson.get_L(A, alignment, helper_poisson.get_Salpha(A), N, T)

        # Reorder L according to latent mapping
        invperm = np.argsort(latent)
        L_reordered = L[np.ix_(invperm, invperm)]
        L_values.append(L_reordered)

    # Average across all L-matrices
    inferred_blueprint = np.mean(np.stack(L_values), axis=0)
    
    return inferred_blueprint

def analysis_ROC(alpha, L, A, t_out, all_labels):
    #y_score will be the probability of each edge having a synapse (1-p(0|Lij, alpha_t))
    Aijt = 0
    mat_probaSynapse = 1-(np.e**(-alpha*L))*((alpha*L)**Aijt)/(math.factorial(Aijt)) #matrix with 1-p(0)

    #proba vs real synapses (heatmaps):
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))  # 1 fila, 2 columnes
    im0 = axs[0].imshow(mat_probaSynapse)
    plt.colorbar(im0, ax=axs[0])
    axs[0].set_title("p(synapse)")
    im1 = axs[1].imshow(A[-1] > 0)
    plt.colorbar(im1, ax=axs[1])
    axs[1].set_title("real A2")
    plt.tight_layout()
    plt.show()

    
    #roc curve (with our model and with the network t-1):
    binary_A2 = (A[t_out] >= 1).astype(int)
    y_true = binary_A2.flatten()
    
    #1) mat_probaSynapse as prediction:
    y_scores_proba = mat_probaSynapse.flatten()
    fpr_proba, tpr_proba, _ = roc_curve(y_true, y_scores_proba)
    roc_auc_proba = auc(fpr_proba, tpr_proba)
    
    #2) counts of A[t-1] as predictors:
    Aprev = A[t_out-1]
    y_scores_aprev = Aprev.flatten()
    fpr_aprev, tpr_aprev, _ = roc_curve(y_true, y_scores_aprev)
    roc_auc_aprev = auc(fpr_aprev, tpr_aprev)

    #plot:
    f = plt.figure(figsize=(5, 5))
    plt.plot(fpr_proba, tpr_proba, color='darkorange', lw=2,
             label=f'ROC curve (p(synapse), {all_labels[t_out]}), AUC = {roc_auc_proba:.3f}')
    plt.plot(fpr_aprev, tpr_aprev, color='darkblue', lw=2,
             label=f'ROC curve ( {all_labels[t_out-1]},  {all_labels[t_out]}), AUC = {roc_auc_aprev:.3f}')
    plt.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--')  # línia d'atzar
    
    ax1 = plt.gca()
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC curves - predicting {all_labels[t_out]}')
    plt.legend(loc='lower right')
    plt.grid(False)
    ax1.set_aspect('equal', adjustable='box')  # quadrat
    ax1.set_box_aspect(1)
    ax1.spines.right.set_visible(False)
    ax1.spines.top.set_visible(False)
    plt.tight_layout()
    plt.show()

    roc_results = {"fpr_model": fpr_proba, "tpr_model": tpr_proba, "fpr_aprev": fpr_aprev, "tpr_aprev": tpr_aprev,
                  "auc_model": roc_auc_proba, "auc_aprev": roc_auc_aprev}
    return roc_results

def compare_predictedVSreal(countsDF, bins=15):
    connectome_labels = countsDF["Connectome"].drop_duplicates().tolist()
    fig_list = []
    for con_label in connectome_labels:
        connectomeDF = countsDF[countsDF["Connectome"] == con_label]

        max_real = connectomeDF["RealCounts"].max()
        length_bin = max_real / bins

        connectomes_id = []
        mean_GW = []
        std_GW = []
        var_GW = []
        mean_RW = []
        std_RW = []
        err_RW = []

        for b in range(bins):
            binnedDF = connectomeDF[
                (connectomeDF["RealCounts"] > b * length_bin) &
                (connectomeDF["RealCounts"] <= (b + 1) * length_bin)
            ]
            sample_size = len(binnedDF)
            if sample_size == 0:
                continue

            connectomes_id.append(con_label)
            mean_GW.append(binnedDF["ModelRates"].mean())
            std_GW.append(binnedDF["ModelRates"].std())
            var_GW.append(binnedDF["ModelRates"].var())
            mean_RW.append(binnedDF["RealCounts"].mean())
            std_RW.append(binnedDF["RealCounts"].std())
            err_RW.append(std_RW[-1] / (sample_size ** 0.5))

        data = pd.DataFrame({
            "connectome": connectomes_id,
            "mean_RealCounts": mean_RW,
            "mean_ModelRates": mean_GW,
            "std_RealCounts": std_RW,
            "var_ModelRates": var_GW,
            "std_ModelRates": std_GW,
            "err_RealCounts": err_RW
        })

        f = plt.figure(figsize=(5, 5)) 
        plt.errorbar(data['mean_RealCounts'], data['mean_ModelRates'],
                     xerr=data['std_RealCounts'], yerr=data['std_ModelRates'],
                     fmt="o", markersize=5)

        ax1 = plt.gca()
        ax1.set_xlabel(r"Real counts", fontsize=10)
        ax1.set_ylabel(r"Predicted model rates", fontsize=10)        
        xmin, xmax = ax1.get_xlim()
        ymin, ymax = ax1.get_ylim()
        max_limit = max(xmax, ymax)
        ax1.set_xlim([xmin, max_limit])
        ax1.set_ylim([ymin, max_limit])
        ax1.plot([ymin, max_limit], [ymin, max_limit], ls="--", color="#bababa", zorder=0)
        ax1.set_title(f'{con_label}', fontsize=12, loc='center')
        ax1.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
        ax1.yaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
        ax1.xaxis.set_tick_params(labelsize=8)
        ax1.yaxis.set_tick_params(labelsize=8)
        ax1.set_aspect('equal', adjustable='box')  
        ax1.set_box_aspect(1)  
        ax1.spines.right.set_visible(False)
        ax1.spines.top.set_visible(False)
        plt.tight_layout()
        plt.show()

        fig_list.append(f)
    return(fig_list)

## Infer alignment from connectomes up to t_study (not included)

In [ ]:
all_con_labels = ["L1.1", "L1.2", "L1.3", "L1.4", "L2", "L3", "A.1", "A.2"]
ages = np.array([ 0.2,  5. ,  8. , 16. , 23. , 27. , 45. , 45.2]) #from Witvliet et al.
beta_spacing = 1.005
numRuns = 5

In [ ]:
A = helper_networks.load_networks(data_folder="data/processed/celegans/", net_ids=[1,2,3,4,5,6,7,8])
T,N,_ = A.shape

neuron_metadata = pd.read_csv("data/processed/celegans/neuron_metadata.csv", index_col=0) 
groups = np.array(neuron_metadata["location"])

In [ ]:
# ALIGN AND INFER BLUEPRINT WITH ALL CONNECTOMES UNTIL CONNECTOME t_study (not included)

for t_study in range(2, T):
    alignment_results = alignment_withprevious(A, t_study, groups, numRuns) #it'll automatically store the results

## Prediction target: next connectome

In [ ]:
#WE'VE ALIGNED UNTIL CONNECTOME t_study, NOW PREDICT t_study

countsDF = pd.DataFrame(columns=["RealCounts", "ModelRates", "Connectome"])
rocDF = pd.DataFrame(columns=["t_pred","auc_model", "auc_previous",  "tpr_model", "fpr_model", "tpr_previous", "fpr_previous"])

for t_study in range(2, T):
    print(f"*********************************")
    print(f"t_study={t_study} ******************")
    print(f"*********************************")

    subset_A = remove_At_onwards(A, t_study)
    
    alignment_results = []
    for run in range(numRuns):
        with open(f"connectomes_upto_{t_study}_run{run}.pkl", "rb") as file:
            results = pickle.load(file)
        alignment_results.append(results)
        
    S = len(alignment_results[run]["Samples"])
    ensemble = np.zeros((S*numRuns, t_study, N)) 
    for run in range(numRuns):
        run_samples = alignment_results[run]["Samples"]
        ensemble[run*S:(run+1)*S, :, :] = run_samples
    sampled_L = infer_blueprint(subset_A, ensemble)
    
    #alpha interpolated from the previous information (linearly)
    alphas_previous = helper_poisson.get_alpha(subset_A, 
                                               helper_poisson.get_ST(subset_A))
    ages_previous = ages[:t_study]
    coef = np.polyfit(ages_previous, alphas_previous, 1)
    interp_alpha = np.polyval(coef, ages[t_study])
    modelinf_alpha = helper_poisson.get_St(A, t_study)/max(np.sum(x) for x in subset_A) #St/ST
    print(f"Interpolated alpha vs model-inferred alpha: {interp_alpha} // {modelinf_alpha}")

    #predicted rates:
    modelrates = interp_alpha*sampled_L

    ## If one wishes to generate/predict counts:
    # generated_fromAlignment = []
    # num_generated = 100
    # for _ in range(num_generated):
    #     onedraw = helper_poisson.synthesize_networks(sampled_L, [interp_alpha]) #or sampled_L if we want to see the blueprint from the alignment
    #     generated_fromAlignment.append(onedraw)

    #comparing predicted and real:
    #from alignment:
    newDF = pd.DataFrame({
        "RealCounts": A[t_study].flatten(),
        "ModelRates": modelrates.flatten(),
        "Connectome": [all_con_labels[t_study]] * (N * N)
    })
    countsDF = pd.concat([countsDF, newDF], ignore_index=True)
    compare_predictedVSreal(newDF, 15)

    #auroc, binary:
    roc_results = analysis_ROC(interp_alpha, sampled_L, A, t_study, all_con_labels)
    new_roc = {"t_pred": all_con_labels[t_study], "auc_model": roc_results["auc_model"],
               "auc_previous": roc_results["auc_aprev"], "tpr_model": roc_results["tpr_model"], "fpr_model": roc_results["fpr_model"],
              "tpr_previous": roc_results["tpr_aprev"], "fpr_previous": roc_results["fpr_aprev"]}
    rocDF = pd.concat([rocDF, pd.DataFrame([new_roc])], ignore_index=True)
    rocDF

## Prediction target: last connectome A.2

In [ ]:
#WE'VE ALIGNED UNTIL CONNECTOME t_study, NOW PREDICT A.2 (last connectome)

countsDF = pd.DataFrame(columns=["RealCounts", "ModelRates", "Connectome"])
rocDF = pd.DataFrame(columns=["t_pred","auc_model", "auc_previous",  "tpr_model", "fpr_model", "tpr_previous", "fpr_previous"])

for t_study in range(2, T):
    print(f"*********************************")
    print(f"t_study={t_study} ******************")
    print(f"*********************************")

    subset_A = remove_At_onwards(A, t_study)
    
    alignment_results = []
    for run in range(numRuns):
        with open(f"connectomes_upto_{t_study}_run{run}.pkl", "rb") as file:
            results = pickle.load(file)
        alignment_results.append(results)
        
    S = len(alignment_results[run]["Samples"])
    ensemble = np.zeros((S*numRuns, t_study, N)) 
    for run in range(numRuns):
        run_samples = alignment_results[run]["Samples"]
        ensemble[run*S:(run+1)*S, :, :] = run_samples
    sampled_L = infer_blueprint(subset_A, ensemble)
    
    #alpha interpolated from the previous information (linearly)
    alphas_previous = helper_poisson.get_alpha(subset_A, 
                                               helper_poisson.get_ST(subset_A))
    ages_previous = ages[:t_study]
    coef = np.polyfit(ages_previous, alphas_previous, 1)
    interp_alpha = np.polyval(coef, ages[-1])
    modelinf_alpha = helper_poisson.get_St(A, t_study)/max(np.sum(x) for x in subset_A) #St/ST
    print(f"Interpolated alpha vs model-inferred alpha: {interp_alpha} // {modelinf_alpha}")

    #predicted rates:
    modelrates = interp_alpha*sampled_L

    ## If one wishes to generate/predict counts:
    # generated_fromAlignment = []
    # num_generated = 100
    # for _ in range(num_generated):
    #     onedraw = helper_poisson.synthesize_networks(sampled_L, [interp_alpha]) #or sampled_L if we want to see the blueprint from the alignment
    #     generated_fromAlignment.append(onedraw)

    #comparing predicted and real:
    #from alignment:
    newDF = pd.DataFrame({
        "RealCounts": A[t_study].flatten(),
        "ModelRates": modelrates.flatten(),
        "Connectome": [all_con_labels[t_study]] * (N * N)
    })
    countsDF = pd.concat([countsDF, newDF], ignore_index=True)
    compare_predictedVSreal(newDF, 15)

    #auroc, binary:
    roc_results = analysis_ROC(interp_alpha, sampled_L, A, -1, all_con_labels)
    new_roc = {"t_pred": all_con_labels[t_study], "auc_model": roc_results["auc_model"],
               "auc_previous": roc_results["auc_aprev"], "tpr_model": roc_results["tpr_model"], "fpr_model": roc_results["fpr_model"],
              "tpr_previous": roc_results["tpr_aprev"], "fpr_previous": roc_results["fpr_aprev"]}
    rocDF = pd.concat([rocDF, pd.DataFrame([new_roc])], ignore_index=True)
    rocDF